In [1]:
!pip install psycopg2-binary sqlalchemy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 42.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

pd.set_option('display.max_columns', None)

NEON_CONNECTION_STRING = " "

engine = create_engine(NEON_CONNECTION_STRING)
print("Connected ✅")

Connected ✅


In [4]:
flights = pd.read_sql("SELECT * FROM flights;", engine)
delays = pd.read_sql("SELECT * FROM delays;", engine)
carriers = pd.read_sql("SELECT * FROM carriers;", engine)
airports = pd.read_sql("SELECT * FROM airports;", engine)

print(f"flights:  {flights.shape}")
print(f"delays:   {delays.shape}")
print(f"carriers: {carriers.shape}")
print(f"airports: {airports.shape}")

flights:  (480000, 13)
delays:   (480000, 7)
carriers: (14, 2)
airports: (351, 3)


In [5]:
df = flights.merge(delays, on='flight_id', how='left')
df = df.merge(carriers, on='carrier_code', how='left')

origin_airports = airports.rename(columns={'airport_code': 'origin', 'city': 'origin_city', 'state': 'origin_state'})
df = df.merge(origin_airports, on='origin', how='left')

dest_airports = airports.rename(columns={'airport_code': 'dest', 'city': 'dest_city', 'state': 'dest_state'})
df = df.merge(dest_airports, on='dest', how='left')

print(f"Final merged shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Final merged shape: 480,000 rows x 24 columns


,flight_id,carrier_code,flight_number,flight_date,origin,dest,dep_delay_min,arr_delay_min,cancelled,cancellation_code,diverted,air_time,distance,delay_id,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,carrier_name,origin_city,origin_state,dest_city,dest_state
0,1,OO,6279,2025-01-29,ORD,BNA,0.0,0.0,0,N,0,54.0,409.0,1,0.0,0.0,0.0,0.0,0.0,OO,"Chicago, IL",IL,"Nashville, TN",TN
1,2,AS,700,2025-01-13,SEA,PHX,0.0,0.0,0,N,0,136.0,1107.0,2,0.0,0.0,0.0,0.0,0.0,AS,"Seattle, WA",WA,"Phoenix, AZ",AZ
2,3,DL,2970,2025-01-20,MSP,DFW,14.0,0.0,0,N,0,119.0,852.0,3,0.0,0.0,0.0,0.0,0.0,DL,"Minneapolis, MN",MN,"Dallas/Fort Worth, TX",TX
3,4,WN,1289,2025-01-17,BWI,DAL,18.0,0.0,0,N,0,175.0,1209.0,4,0.0,0.0,0.0,0.0,0.0,WN,"Baltimore, MD",MD,"Dallas, TX",TX
4,5,G4,1808,2025-01-18,ABE,SFB,0.0,0.0,0,N,0,136.0,882.0,5,0.0,0.0,0.0,0.0,0.0,G4,"Allentown/Bethlehem/Easton, PA",PA,"Sanford, FL",FL


#Build Target Variable & Filter Dataset

In [6]:
model_df = df.copy()

# Convert date and extract time features
model_df['flight_date'] = pd.to_datetime(model_df['flight_date'], errors='coerce')
model_df['month'] = model_df['flight_date'].dt.month
model_df['day_of_week'] = model_df['flight_date'].dt.day_name()
model_df['day_of_month'] = model_df['flight_date'].dt.day
model_df['quarter'] = model_df['flight_date'].dt.quarter

# Exclude cancelled flights — delay isn't meaningfully defined for them
model_df = model_df[model_df['cancelled'] == 0].copy()

# Define target: DOT standard delay threshold = 15 minutes
model_df['is_delayed'] = (model_df['arr_delay_min'] >= 15).astype(int)

print(f"Total flights for modeling: {len(model_df):,}")
print(f"Delayed: {model_df['is_delayed'].sum():,} ({model_df['is_delayed'].mean()*100:.2f}%)")
print(f"On-time: {(model_df['is_delayed']==0).sum():,} ({(1-model_df['is_delayed'].mean())*100:.2f}%)")

Total flights for modeling: 472,876
Delayed: 104,639 (22.13%)
On-time: 368,237 (77.87%)


#Feature Engineering

In [7]:
from sklearn.preprocessing import LabelEncoder

# Select pre-flight features only (no leakage)
feature_cols = ['carrier_code', 'origin', 'dest', 'distance', 'month', 'day_of_week', 'quarter', 'day_of_month']

model_data = model_df[feature_cols + ['is_delayed']].copy()

# Encode categorical columns
encoders = {}
for col in ['carrier_code', 'origin', 'dest', 'day_of_week']:
    le = LabelEncoder()
    model_data[col] = le.fit_transform(model_data[col])
    encoders[col] = le  # save encoder in case we need to decode later

model_data.head()

,carrier_code,origin,dest,distance,month,day_of_week,quarter,day_of_month,is_delayed
0,10,241,42,409.0,1,6,1,29,0
1,1,298,252,1107.0,1,1,1,13,0
2,3,229,90,852.0,1,1,1,20,0
3,12,55,84,1209.0,1,0,1,17,0
4,5,0,299,882.0,1,2,1,18,0


#Train/Test Split

In [8]:
from sklearn.model_selection import train_test_split

X = model_data.drop('is_delayed', axis=1)
y = model_data['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train delay rate: {y_train.mean()*100:.2f}%")
print(f"Test delay rate: {y_test.mean()*100:.2f}%")

Train shape: (378300, 8)
Test shape: (94576, 8)
Train delay rate: 22.13%
Test delay rate: 22.13%


# Train Random Forest (Baseline, No Tuning Yet)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

start = time.time()

rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)
y_pred = rf_baseline.predict(X_test)

print(f"Training time: {time.time()-start:.1f} seconds")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['On-Time', 'Delayed'])}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

Training time: 43.8 seconds

Accuracy: 59.79%

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.84      0.60      0.70     73648
     Delayed       0.30      0.60      0.40     20928

    accuracy                           0.60     94576
   macro avg       0.57      0.60      0.55     94576
weighted avg       0.72      0.60      0.63     94576


Confusion Matrix:
[[43921 29727]
 [ 8299 12629]]


#GridSearchCV Tuning

In [10]:
from sklearn.model_selection import GridSearchCV
import time

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5]
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_search.fit(X_train, y_train)
print(f"\nGrid search time: {(time.time()-start)/60:.1f} minutes")

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_*100:.2f}%")

Fitting 3 folds for each of 24 candidates, totalling 72 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Grid search time: 63.1 minutes

Best params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV accuracy: 74.84%


#Evaluate Tuned Model on Test Set

In [11]:
best_rf = grid_search.best_estimator_

y_pred_tuned = best_rf.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_tuned, target_names=['On-Time', 'Delayed'])}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred_tuned)}")

Test Accuracy: 74.18%

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.81      0.87      0.84     73648
     Delayed       0.39      0.29      0.33     20928

    accuracy                           0.74     94576
   macro avg       0.60      0.58      0.59     94576
weighted avg       0.72      0.74      0.73     94576


Confusion Matrix:
[[64087  9561]
 [14858  6070]]


In [12]:
# Re-derive from model_df (before encoding) to add new features cleanly
model_df['is_weekend'] = model_df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Route-level historical delay rate (computed from training data only, to avoid leakage into test set)
model_df['route'] = model_df['origin'] + '_' + model_df['dest']

print("New features added: is_weekend, route")
model_df[['day_of_week', 'is_weekend', 'route']].head()

New features added: is_weekend, route


,day_of_week,is_weekend,route
0,Wednesday,0,ORD_BNA
1,Monday,0,SEA_PHX
2,Monday,0,MSP_DFW
3,Friday,0,BWI_DAL
4,Saturday,1,ABE_SFB


#Add Stronger Features

In [14]:
feature_cols = ['carrier_code', 'origin', 'dest', 'distance', 'month', 'day_of_week',
                 'quarter', 'day_of_month', 'is_weekend', 'route']

model_data = model_df[feature_cols + ['is_delayed']].copy()

from sklearn.model_selection import train_test_split

X = model_data.drop('is_delayed', axis=1)
y = model_data['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (378300, 10)
Test shape: (94576, 10)


#Compute Historical Delay Rates

In [15]:
# Compute route-level delay rate using ONLY training data
route_delay_rate = X_train.join(y_train).groupby('route')['is_delayed'].mean()
overall_delay_rate = y_train.mean()  # fallback for routes not seen in training

X_train['route_delay_rate'] = X_train['route'].map(route_delay_rate)
X_test['route_delay_rate'] = X_test['route'].map(route_delay_rate).fillna(overall_delay_rate)

# Compute carrier-level delay rate using ONLY training data
carrier_delay_rate = X_train.join(y_train).groupby('carrier_code')['is_delayed'].mean()

X_train['carrier_delay_rate'] = X_train['carrier_code'].map(carrier_delay_rate)
X_test['carrier_delay_rate'] = X_test['carrier_code'].map(carrier_delay_rate).fillna(overall_delay_rate)

print(f"Routes seen in training: {X_train['route'].nunique():,}")
print(f"Routes in test not seen in training: {(~X_test['route'].isin(route_delay_rate.index)).sum():,}")
print(f"\nSample route delay rates:")
print(route_delay_rate.sort_values(ascending=False).head())

Routes seen in training: 6,489
Routes in test not seen in training: 85

Sample route delay rates:
route
OKC_VPS    1.0
XNA_MSP    1.0
VRB_BOS    1.0
PNS_AUS    1.0
PSP_EWR    1.0
Name: is_delayed, dtype: float64


In [16]:
route_counts = X_train.join(y_train).groupby('route').size()
print(f"Routes with only 1 flight in training: {(route_counts == 1).sum():,}")
print(f"Routes with fewer than 10 flights: {(route_counts < 10).sum():,}")
print(f"Routes with 100+ flights: {(route_counts >= 100).sum():,}")

Routes with only 1 flight in training: 283
Routes with fewer than 10 flights: 1,507
Routes with 100+ flights: 1,187


#Smooth the Historical Rates

In [17]:
def smoothed_rate(group_counts, group_means, overall_rate, k=20):
    """
    k = smoothing strength. Higher k = more shrinkage toward overall average
    for low-volume groups. k=20 means a group needs ~20+ flights before its
    own rate starts to dominate over the overall average.
    """
    return (group_counts * group_means + k * overall_rate) / (group_counts + k)

# --- Route-level smoothed rate ---
route_stats = X_train.join(y_train).groupby('route')['is_delayed'].agg(['mean', 'count'])
route_stats['smoothed_rate'] = smoothed_rate(route_stats['count'], route_stats['mean'], overall_delay_rate, k=20)

X_train['route_delay_rate'] = X_train['route'].map(route_stats['smoothed_rate'])
X_test['route_delay_rate'] = X_test['route'].map(route_stats['smoothed_rate']).fillna(overall_delay_rate)

# --- Carrier-level smoothed rate (carriers have plenty of data each, but let's be consistent) ---
carrier_stats = X_train.join(y_train).groupby('carrier_code')['is_delayed'].agg(['mean', 'count'])
carrier_stats['smoothed_rate'] = smoothed_rate(carrier_stats['count'], carrier_stats['mean'], overall_delay_rate, k=20)

X_train['carrier_delay_rate'] = X_train['carrier_code'].map(carrier_stats['smoothed_rate'])
X_test['carrier_delay_rate'] = X_test['carrier_code'].map(carrier_stats['smoothed_rate']).fillna(overall_delay_rate)

print("Before vs after smoothing — sample low-volume routes:")
comparison = route_stats.sort_values('count').head(10)
print(comparison)

Before vs after smoothing — sample low-volume routes:
         mean  count  smoothed_rate
route                              
ABQ_PDX   0.0      1       0.210745
ABQ_DTW   0.0      1       0.210745
XNA_MSP   1.0      1       0.258364
MCI_SRQ   0.0      1       0.210745
ORF_SFB   0.0      1       0.210745
ORD_GUC   0.0      1       0.210745
ORD_GTF   0.0      1       0.210745
MSN_MCO   0.0      1       0.210745
MSN_LAX   0.0      1       0.210745
FNT_EWR   0.0      1       0.210745


#Drop Raw Categorical Columns We No Longer Need, Encode What's Left

In [18]:
from sklearn.preprocessing import LabelEncoder

# Drop raw 'route' — its signal is now captured numerically by route_delay_rate
X_train_final = X_train.drop(columns=['route']).copy()
X_test_final = X_test.drop(columns=['route']).copy()

# Encode remaining categoricals
encoders = {}
for col in ['carrier_code', 'origin', 'dest', 'day_of_week']:
    le = LabelEncoder()
    X_train_final[col] = le.fit_transform(X_train_final[col])
    # Use the SAME encoder fit on train to transform test — handle unseen labels safely
    X_test_final[col] = X_test_final[col].map(lambda x: x if x in le.classes_ else 'UNKNOWN')
    le_classes = list(le.classes_)
    if 'UNKNOWN' not in le_classes:
        le_classes.append('UNKNOWN')
    le.classes_ = np.array(le_classes)
    X_test_final[col] = le.transform(X_test_final[col])
    encoders[col] = le

print(f"Final train shape: {X_train_final.shape}")
print(f"Final test shape: {X_test_final.shape}")
X_train_final.head()

Final train shape: (378300, 11)
Final test shape: (94576, 11)


,carrier_code,origin,dest,distance,month,day_of_week,quarter,day_of_month,is_weekend,route_delay_rate,carrier_delay_rate
250426,12,318,70,575.0,7,4,3,10,0,0.205983,0.212130
356630,12,89,205,1547.0,9,0,3,5,0,0.244608,0.212130
218831,10,299,313,372.0,6,0,2,27,0,0.187234,0.206373
382702,12,218,205,1066.0,10,4,4,23,0,0.246724,0.212130
75113,11,229,161,305.0,2,1,1,17,0,0.191714,0.214386


#Retrain Random Forest with New Features

In [19]:
rf_v2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

start = time.time()
rf_v2.fit(X_train_final, y_train)
y_pred_v2 = rf_v2.predict(X_test_final)

print(f"Training time: {time.time()-start:.1f} seconds")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_v2)*100:.2f}%")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_v2, target_names=['On-Time', 'Delayed'])}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred_v2)}")

Training time: 216.9 seconds

Accuracy: 73.95%

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.81      0.87      0.84     73648
     Delayed       0.38      0.27      0.31     20928

    accuracy                           0.74     94576
   macro avg       0.59      0.57      0.58     94576
weighted avg       0.71      0.74      0.72     94576


Confusion Matrix:
[[64333  9315]
 [15325  5603]]


I deliberately excluded any feature that wouldn't be known before departure — no delay-cause breakdowns, no departure delay, no actual air time. With that constraint, the model plateaus around 74% accuracy regardless of tuning or added engineered features, which tells us that pre-flight contextual features (carrier, route, season) explain a meaningful but limited portion of delay variance. The remaining ~26% is likely driven by same-day operational factors — weather, air traffic congestion, aircraft turnaround — that aren't available as pre-flight signals in this dataset.